In [3]:

import requests, json

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
HEADERS = {"Authorization": f"Bearer {TOKEN}"}
BASE = "https://nzcwegq7.api.sanity.io/v2021-06-07/data/query/production"

def sq(query):
    r = requests.get(BASE, params={"query": query}, headers=HEADERS, timeout=30)
    return r.json().get("result", [])

# Wszystkie kategorie z pełną ścieżką
cats = sq('*[_type=="category"]{_id, name, "slug":slug.current, "parentSlug":parent->slug.current, "parentName":parent->name, "rootSlug":parent->parent->slug.current}')
print(f"Łączna liczba kategorii: {len(cats)}")
print("\nKategorie główne (bez rodzica):")
roots = [c for c in cats if not c.get('parentSlug')]
for c in sorted(roots, key=lambda x: x['name']):
    print(f"  {c['name']} → slug: {c['slug']}")

print("\nPodkategorie (mają rodzica):")
subs = [c for c in cats if c.get('parentSlug') and not c.get('rootSlug')]
print(f"  Liczba podkategorii: {len(subs)}")
for c in sorted(subs, key=lambda x: x.get('parentName','') + x['name'])[:30]:
    print(f"  [{c.get('parentName','')}] → {c['name']} ({c['slug']})")


Łączna liczba kategorii: 321

Kategorie główne (bez rodzica):
  Chemia budowlana → slug: chemia-budowlana
  Dachy → slug: dachy
  Farby i rozpuszczalniki → slug: farby-i-rozpuszczalniki
  Izolacje → slug: izolacje
  Narzędzia i mocowania → slug: narzedzia-i-mocowania
  Oświetlenie → slug: oswietlenie
  Pozostałe → slug: pozostale
  Płytki → slug: plytki
  Stropy i ściany → slug: stropy-i-sciany
  Sucha zabudowa → slug: sucha-zabudowa
  Sufity podwieszane → slug: sufity-podwieszane

Podkategorie (mają rodzica):
  Liczba podkategorii: 72
  [Chemia budowlana] → Dodatki do zapraw i betonu (dodatki-do-zapraw-i-betonu)
  [Chemia budowlana] → Gipsy i gładzie (gipsy-i-gladzie)
  [Chemia budowlana] → Grunty (grunty)
  [Chemia budowlana] → Kleje (kleje)
  [Chemia budowlana] → Kotwy chemiczne (kotwy-chemiczne)
  [Chemia budowlana] → Piany montażowe (piany-montazowe)
  [Chemia budowlana] → Powłoki epoksydowe (powloki-epoksydowe)
  [Chemia budowlana] → Spoiny (spoiny)
  [Chemia budowlana] → Tynki (

In [7]:

# Pokaż resztę podkategorii
print("Pozostałe podkategorie:")
for c in sorted(subs, key=lambda x: x.get('parentName','') + x['name'])[30:]:
    print(f"  [{c.get('parentName','')}] → {c['name']} ({c['slug']})")

sub_subs = [c for c in cats if c.get('rootSlug')]
print(f"\nPodpodkategorie (poziom 3): {len(sub_subs)}")
for c in sorted(sub_subs, key=lambda x: x['name'])[:20]:
    print(f"  {c['name']} ({c['slug']})")


Pozostałe podkategorie:
  [Izolacje] → Akcesoria do izolacji (akcesoria-do-izolacji)
  [Izolacje] → Folie (folie)
  [Izolacje] → Hydroizolacje (hydroizolacje)
  [Izolacje] → Izolacje budowlane (izolacje-budowlane)
  [Izolacje] → Izolacje techniczne (izolacje-techniczne)
  [Izolacje] → Płyty XPS (plyty-xps)
  [Izolacje] → Styropiany (styropiany)
  [Izolacje] → Wełny (welny)
  [Narzędzia i mocowania] → Akcesoria malarskie i tynkarskie (akcesoria-malarskie-i-tynkarskie)
  [Narzędzia i mocowania] → Akcesoria murarskie (akcesoria-murarskie)
  [Narzędzia i mocowania] → Artykuły ścierne (artykuly-scierne)
  [Narzędzia i mocowania] → Elektronarzędzia (elektronarzedzia)
  [Narzędzia i mocowania] → Elementy mocujące uniwersalne (elementy-mocujace-uniwersalne)
  [Narzędzia i mocowania] → Kielnie (kielnie)
  [Narzędzia i mocowania] → Narzędzia budowlane (narzedzia-budowlane)
  [Narzędzia i mocowania] → Narzędzia do cięcia (narzedzia-do-ciecia)
  [Narzędzia i mocowania] → Narzędzia i akcesoria glaz

In [11]:

# Szukam produktów w kategorii "tynki" które brzmią jak wełna/izolacja/styropian
tynki_ids = [c['_id'] for c in cats if c['slug'] == 'tynki']
print(f"ID kategorii tynki: {tynki_ids}")

# Pobierz produkty z kategorii tynki i sprawdź ich nazwy
q = '''*[_type=="product" && references("%s") && !(name match "P-*")]{
  _id, name, "catSlug":category->slug.current, "catName":category->name
}[0..99]''' % (tynki_ids[0] if tynki_ids else 'xxx')

tynki_prods = sq(q)
print(f"\nLiczba produktów w 'tynki' (pierwsze 100): {len(tynki_prods)}")

# Szukaj podejrzanych (wełna, izolacja, styropian w tynkach)
suspicious_keywords = ['wełn', 'welna', 'styropian', 'izolac', 'rockwool', 'isover', 'ursa', 'knauf insul', 'mineral', 'glasswool']
suspicious = [p for p in tynki_prods if any(k.lower() in p['name'].lower() for k in suspicious_keywords)]
print(f"Podejrzane w 'tynki': {len(suspicious)}")
for p in suspicious[:20]:
    print(f"  {p['name']}")

# Sprawdź też pozostałe produkty - szukaj "wełna" w złych kategoriach
print("\n--- Szukam 'wełna' w kategoriach innych niż welny/izolacje ---")
welna_prods = sq('*[_type=="product" && name match "*Wełna*" && !(name match "P-*")]{_id,name,"catSlug":category->slug.current,"catName":category->name}[0..50]')
print(f"Produkty z 'Wełna' w nazwie: {len(welna_prods)}")
wrong_cat_welna = [p for p in welna_prods if p.get('catSlug') not in ['welny','izolacje','izolacje-budowlane','akcesoria-do-izolacji']]
print(f"Z nich w złej kategorii: {len(wrong_cat_welna)}")
for p in wrong_cat_welna[:20]:
    print(f"  [{p.get('catSlug')}] {p['name']}")


ID kategorii tynki: ['cat-tynki']

Liczba produktów w 'tynki' (pierwsze 100): 0
Podejrzane w 'tynki': 0

--- Szukam 'wełna' w kategoriach innych niż welny/izolacje ---


Produkty z 'Wełna' w nazwie: 51
Z nich w złej kategorii: 51
  [welny-fasadowe] Wełna fasadowa ROCKWOOL Frontrock MAX E 15cm
  [welny-fasadowe] Wełna fasadowa Isover Polterm Uni 100mm
  [welny-fasadowe] wełna Isover Super Vent Plus 150mm 2,16m2
  [welny-fasadowe] wełna fasadowa Isover Fasoterm 35 200mm
  [izolacje-hvac] Wełna mineralna Rockwool Klimafix 20 cm Opakowanie 10 m2
  [izolacje-hvac] Wełna mineralna Rockwool Lamella Mat Alu rolka 10m2 grubość 20mm
  [welny-fasadowe] wełna fasadowa Rockwool Ventirock 50mm 4,8m2/8 szt.
  [welny-fasadowe] Wełna fasadowa Rockwool Frontrock S 30 cm
  [welny-fasadowe] Wełna fasadowa Rockwool Ventirock F Plus 80mm
  [welny-fasadowe] Wełna fasadowa Ursa Vento 34 160mm opakowanie 3m2
  [welny-do-poddaszy] wełna mineralna Eurowool M17 L035 50x10000x1200
  [welny-fasadowe] wełna fasadowa Polterm Uni 150mm
  [welny-do-poddaszy] wełna Rockwool Superrock Premium L034 Opakowanie 4,88 m²
  [izolacje-przemyslowe] wełna Orstech 30mm
  [welny-do-poddaszy] Wełna 

In [15]:

# Sprawdź parent kategorii dla welny-fasadowe itd.
welny_cat_slugs = list(set([p['catSlug'] for p in wrong_cat_welna]))
print("Unikalne kategorie wełny:", welny_cat_slugs)

# Pobierz ich parent
for slug in welny_cat_slugs[:10]:
    details = sq(f'*[_type=="category" && slug.current=="{slug}"]{{name,"parentSlug":parent->slug.current,"parentName":parent->name,"grandSlug":parent->parent->slug.current}}[0]')
    if details:
        d = details[0]
        print(f"  {slug} → parent: {d.get('parentSlug')} ({d.get('parentName')}) → grandparent: {d.get('grandSlug')}")

# Szukaj produktów z grubością w nazwie i sprawdź parametry
print("\n=== MISMATCH grubości nazwa vs parametry ===")
# Szukaj np. "16cm" lub "160mm" w nazwie
thick_prods = sq('''*[_type=="product" && (name match "*16cm*" || name match "*160mm*") && !(name match "P-*") && defined(technicalSpec) && count(technicalSpec)>0]{
  _id, name, technicalSpec[0..3]
}[0..20]''')

print(f"Produkty z '16cm' lub '160mm' w nazwie: {len(thick_prods)}")
for p in thick_prods[:10]:
    specs = p.get('technicalSpec', [])
    # Szukaj parametru z grubością
    thickness_spec = [s for s in specs if any(k in s.get('label','').lower() for k in ['grub','thick','wymiar','rozmiar'])]
    spec_str = " | ".join([f"{s.get('label')}: {s.get('value')}" for s in specs[:4]])
    print(f"  NAZWA: {p['name']}")
    print(f"  SPEC:  {spec_str}")
    print()


Unikalne kategorie wełny: ['welny-do-poddaszy', 'akcesoria-do-kominow', 'welny-fasadowe', 'izolacje-przemyslowe', 'welny-do-dachow-plaskich', 'izolacje-hvac']


KeyError: 0

In [19]:

import re

# Fix: sq zwraca listę, ale single-object query zwraca dict bezpośrednio
def sq_one(query):
    r = requests.get(BASE, params={"query": query}, headers=HEADERS, timeout=30)
    return r.json().get("result")

# Sprawdź parent kategorii wełny
welny_cat_slugs = ['welny-fasadowe','welny-do-poddaszy','welny-do-dachow-plaskich','izolacje-hvac','izolacje-przemyslowe']
print("=== Hierarchia kategorii wełny ===")
for slug in welny_cat_slugs:
    d = sq_one(f'*[_type=="category" && slug.current=="{slug}"]{{name,"p1":parent->slug.current,"p2":parent->parent->slug.current,"p3":parent->parent->parent->slug.current}}[0]')
    if d:
        path = " > ".join([x for x in [d.get('p3'),d.get('p2'),d.get('p1'),d.get('name')] if x])
        print(f"  {slug}: {path}")

# Szukaj produktów z wymiarami w nazwie - sprawdź spec mismatch
print("\n=== MISMATCH: wymiary w nazwie vs parametry ===")
# Próbka: produkty z "cm" lub "mm" w nazwie
dim_prods = sq('''*[_type=="product" && !(name match "P-*") && defined(technicalSpec) && count(technicalSpec)>0
  && (name match "*cm*" || name match "*mm*")
]{_id, name, technicalSpec[0..5]}[0..200]''')

mismatches = []
for p in dim_prods:
    name = p['name']
    specs = p.get('technicalSpec', [])
    # Wyciągnij wymiary z nazwy
    name_dims = re.findall(r'(\d+(?:[.,]\d+)?)\s*(?:cm|mm)', name.lower())
    # Wyciągnij wymiary z parametrów
    spec_dims = []
    for s in specs:
        val = str(s.get('value',''))
        spec_dims += re.findall(r'(\d+(?:[.,]\d+)?)\s*(?:cm|mm)', val.lower())
    
    if name_dims and spec_dims:
        # Sprawdź czy wymiar z nazwy pojawia się w specach
        found = any(nd in spec_dims for nd in name_dims)
        if not found:
            mismatches.append({
                'id': p['_id'], 'name': name,
                'name_dims': name_dims, 'spec_dims': spec_dims[:5],
                'specs': [(s.get('label',''),s.get('value','')) for s in specs[:4]]
            })

print(f"Produktów z wymiarami w nazwie: {len(dim_prods)}")
print(f"Potencjalnych mismatch: {len(mismatches)}")
print("\nPrzykłady (pierwsze 15):")
for m in mismatches[:15]:
    print(f"  NAZWA: {m['name']}")
    print(f"  Wymiary w nazwie: {m['name_dims']} | W spec: {m['spec_dims']}")
    print(f"  Spec: {m['specs'][:2]}")
    print()


=== Hierarchia kategorii wełny ===


  welny-fasadowe: izolacje > welny > Wełny fasadowe


  welny-do-poddaszy: izolacje > welny > Wełny do poddaszy
  welny-do-dachow-plaskich: izolacje > welny > Wełny do dachów płaskich


  izolacje-hvac: izolacje > izolacje-techniczne > Izolacje HVAC
  izolacje-przemyslowe: izolacje > izolacje-techniczne > Izolacje przemysłowe

=== MISMATCH: wymiary w nazwie vs parametry ===


Produktów z wymiarami w nazwie: 201
Potencjalnych mismatch: 52

Przykłady (pierwsze 15):
  NAZWA: Bloczek Betonowy Komorkowy Betard Z Betonu C12 15 38X24X12 Cm
  Wymiary w nazwie: ['12'] | W spec: ['175', '62,5', '25']
  Spec: [('Typ', 'Bloczek z betonu komórkowego'), ('Zastosowanie', 'Ściany konstrukcyjne, ściany działowe')]

  NAZWA: Wkrety Do Drewna Siniat Nida 3 5 25 Mm Opak 1000 Szt
  Wymiary w nazwie: ['25'] | W spec: ['35']
  Spec: [('Typ', 'Wkręty do drewna'), ('Zastosowanie', 'Montaż elementów drewnianych, sucha zabudowa')]

  NAZWA: Bloczek H H Silver 2 5 500 120 240 590 Mm
  Wymiary w nazwie: ['590'] | W spec: ['59']
  Spec: [('Typ', 'Bloczek z betonu komórkowego'), ('Zastosowanie', 'Ściany nośne i działowe')]

  NAZWA: Plyta Gipsowo Kartonowa Siniat Smart 12 5X1200X2000 Mm Ks
  Wymiary w nazwie: ['2000'] | W spec: ['3000']
  Spec: [('Typ', 'Płyta gipsowo-kartonowa GKB'), ('Zastosowanie', 'Do suchych zabudów wewnętrznych')]

  NAZWA: Bloczek Betonowy Komorkowy Komorkowy Yton

In [23]:

import re, json

# Pobierz WSZYSTKIE produkty z wymiarami (większa próbka)
print("Pobieranie produktów z wymiarami...")
all_dim_prods = []
for offset in [0, 200, 400, 600]:
    batch = sq(f'''*[_type=="product" && !(name match "P-*") && defined(technicalSpec) && count(technicalSpec)>0
    && (name match "*cm*" || name match "*mm*")
    ]{{_id, name, "catSlug":category->slug.current, technicalSpec[0..5]}}[{offset}..{offset+200}]''')
    all_dim_prods.extend(batch)
    if len(batch) < 200:
        break

print(f"Łącznie produktów z wymiarami: {len(all_dim_prods)}")

# Znajdź mismatches
all_mismatches = []
for p in all_dim_prods:
    name = p['name']
    specs = p.get('technicalSpec', [])
    name_dims = set(re.findall(r'\b(\d{2,4})\s*(?:cm|mm)\b', name.lower()))
    spec_dims = set()
    for s in specs:
        spec_dims |= set(re.findall(r'\b(\d{2,4})\s*(?:cm|mm)\b', str(s.get('value','')).lower()))
    
    if name_dims and spec_dims and not (name_dims & spec_dims):
        all_mismatches.append({
            '_id': p['_id'],
            'name': p['name'],
            'catSlug': p.get('catSlug',''),
            'name_dims': sorted(name_dims),
            'spec_dims': sorted(spec_dims),
        })

print(f"Produktów z MISMATCH wymiarów: {len(all_mismatches)}")

# Zapisz do pliku
with open('/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/dim_mismatches.json', 'w', encoding='utf-8') as f:
    json.dump(all_mismatches, f, ensure_ascii=False, indent=2)
print("Zapisano do dim_mismatches.json")

# Pokaz dystrybucję po kategoriach
from collections import Counter
cat_counts = Counter(m['catSlug'] for m in all_mismatches)
print("\nTop kategorie z błędami:")
for cat, cnt in cat_counts.most_common(15):
    print(f"  {cat}: {cnt}")


Pobieranie produktów z wymiarami...


Łącznie produktów z wymiarami: 804
Produktów z MISMATCH wymiarów: 62
Zapisano do dim_mismatches.json

Top kategorie z błędami:
  welny-do-poddaszy: 15
  welny-fasadowe: 15
  laczniki-do-izolacji-fasadowych: 7
  narzedzia-pomiarowe: 5
  welny-do-dachow-plaskich: 4
  profile-do-suchej-zabudowy: 4
  pedzle: 4
  bloczki: 3
  szlifierki: 1
  izolacje-hvac: 1
  wkrety-do-suchej-zabudowy: 1
  akcesoria-do-izolacji: 1
  izolacje-fasadowe: 1


In [27]:

# Sprawdź pola produktów z mismatch - może mają externalId, sku, sourceUrl
with open('/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/dim_mismatches.json') as f:
    mismatches = json.load(f)

# Pobierz pełne dane pierwszych 5 mismatch produktów
for m in mismatches[:5]:
    prod = sq_one(f'*[_id=="{m["_id"]}"][0]')
    if prod:
        keys = [k for k in prod.keys() if k not in ['_id','_type','_rev','_createdAt','_updatedAt']]
        print(f"POLA: {keys}")
        # Pokaz SKU/EAN/external fields
        for k in ['sku','ean','externalId','sourceUrl','externalUrl','productId','catalogId']:
            if k in prod:
                print(f"  {k}: {prod[k]}")
        print(f"  name: {prod.get('name','')}")
        print(f"  brand: {prod.get('brand','')}")
        print()
        break  # Tylko jeden żeby zobaczyć strukturę


POLA: ['brand', 'category', 'description', 'ean', 'featured', 'inStock', 'manufacturerIndex', 'name', 'seoNameUpdated', 'shortDescription', 'sku', 'slug', 'tags', 'technicalSpec']
  sku: P-0006426
  ean: 5902619954055
  name: Welna Mineralna Szklana Ursa Homewall 100 Mm
  brand: {'_ref': 'brand-op375m2', '_type': 'reference'}



In [31]:

import json, requests

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
HEADERS = {"Authorization": f"Bearer {TOKEN}"}
BASE = "https://nzcwegq7.api.sanity.io/v2021-06-07/data/query/production"

def sq(q):
    r = requests.get(BASE, params={"query": q}, headers=HEADERS, timeout=30)
    return r.json().get("result", [])

# Bechcicki.pl NIE ma kategorii "Oświetlenie" jako głównej - sprawdź ile produktów
oswiet = sq('count(*[_type=="product" && references(*[_type=="category" && slug.current=="oswietlenie"]._id) && !(name match "P-*")])')
print(f"Produkty w kategorii 'oswietlenie' (główna): {oswiet}")

# Kategorie w Sanity których NIE MA na bechcicki.pl
bechcicki_cats = {
    "chemia-budowlana","dachy","farby-i-rozpuszczalniki","izolacje",
    "narzedzia-i-mocowania","pozostale","plytki","stropy-i-sciany",
    "sucha-zabudowa","sufity-podwieszane"
}
# Sanity top-level cats
sanity_root = sq('*[_type=="category" && !defined(parent)]{name,"slug":slug.current}')
extra = [c for c in sanity_root if c['slug'] not in bechcicki_cats]
print(f"\nKategorie w Sanity których NIE MA na bechcicki.pl: {[c['slug'] for c in extra]}")

# ===== ANALIZA: jakie kategorie bechcicki.pl mają subcategorie których brak w Sanity? =====
# Pobierz wszystkie subcategorie z bechcicki.pl (ze scrapu)
bechcicki_l2 = {
    # Chemia
    "tynki","kleje","gipsy-i-gladzie","grunty","piany-montazowe","uszczelniacze-i-silikony",
    "zaprawy","spoiny","powloki-epoksydowe","kotwy-chemiczne","srodki-grzybobojcze",
    "srodki-czyszczaco-pielegnacyjne","dodatki-do-zapraw-i-betonu",
    # Dachy
    "pokrycia-dachowe","okna-dachowe-i-akcesoria","fotowoltaika","rynny",
    "zamocowania-dachowe","komunikacja-dachowa","dachy-zielone","zabezpieczenia-przeciwsniegowe",
    # Farby
    "farby-wewnetrzne","farby-elewacyjne","farby-do-drewna","farby-do-metalu",
    "bazy-i-koloranty","farby-specjalistyczne","farby-pozostale","rozpuszczalniki",
    "preparaty-do-chemicznego-oczyszczania-powierzchni",
    # Izolacje
    "styropiany","plyty-xps","welny","izolacje-budowlane","izolacje-techniczne",
    "hydroizolacje","folie","akcesoria-do-izolacji",
    # Narzędzia
    "elementy-mocujace-uniwersalne","akcesoria-malarskie-i-tynkarskie","akcesoria-murarskie",
    "artykuly-scierne","narzedzia-reczne","narzedzia-i-akcesoria-glazurnicze",
    "narzedzia-budowlane","narzedzia-malarskie","elektronarzedzia","narzedzia-pomiarowe",
    "narzedzia-do-ciecia","narzedzia-pozostale",
    # Pozostałe
    "galanteria-betonowa","nawadnianie","stolarka-otworowa","bhp",
    # Płytki
    "plytki-ceramiczne","plytki-dekoracyjne","listwy-i-akcesoria",
    # Stropy i ściany
    "materialy-konstrukcyjne","panele-scienne-i-tapety","schody-i-akcesoria-strychowe","systemy-kominowe",
    # Sucha zabudowa
    "plyty","profile-do-suchej-zabudowy","wieszaki-do-suchej-zabudowy","mocowania-do-suchej-zabudowy",
    "narozniki-i-listwy","tasmy-do-suchej-zabudowy","rewizje",
    # Sufity
    "plyty-sufitowe","profile-do-sufitow-podwieszanych","mocowania-do-sufitow-podwieszanych",
}

sanity_l2 = sq('*[_type=="category" && defined(parent) && !defined(parent->parent)]{"slug":slug.current, name}')
sanity_l2_slugs = {c['slug'] for c in sanity_l2}
missing_in_sanity = bechcicki_l2 - sanity_l2_slugs
extra_in_sanity = sanity_l2_slugs - bechcicki_l2
print(f"\nPodkategorie bechcicki.pl BRAKUJĄCE w Sanity: {sorted(missing_in_sanity)}")
print(f"\nPodkategorie w Sanity KTÓRYCH NIE MA na bechcicki.pl: {sorted(extra_in_sanity)}")


Produkty w kategorii 'oswietlenie' (główna): 0

Kategorie w Sanity których NIE MA na bechcicki.pl: ['oswietlenie']



Podkategorie bechcicki.pl BRAKUJĄCE w Sanity: []

Podkategorie w Sanity KTÓRYCH NIE MA na bechcicki.pl: ['kielnie']


In [35]:

import re, json

# 1. Sprawdź produkty w "tynki" z słowami typowymi dla wełny/styropianu/izolacji
def prods_in_cat(slug, limit=200):
    cat_id_res = sq(f'*[_type=="category" && slug.current=="{slug}"]._id[0]')
    if not cat_id_res:
        return []
    cat_id = cat_id_res
    return sq(f'*[_type=="product" && references("{cat_id}") && !(name match "P-*")]{{_id,name,"catSlug":category->slug.current}}[0..{limit}]')

# Pobierz produkty ze wszystkich podkategorii chemii budowlanej
chemia_subs = ['tynki','kleje','gipsy-i-gladzie','grunty','zaprawy','spoiny']
insulation_words = ['wełn','welna','styropian','rockwool','isover','ursa','paroc','knauf insul','mineral wool','mineralna']

wrong_cat = []
for slug in chemia_subs:
    prods = prods_in_cat(slug)
    for p in prods:
        if any(w.lower() in p['name'].lower() for w in insulation_words):
            wrong_cat.append({'slug': slug, 'name': p['name'], 'id': p['_id']})

print(f"Produkty izolacyjne w chemii budowlanej: {len(wrong_cat)}")
for p in wrong_cat[:20]:
    print(f"  [{p['slug']}] {p['name']}")

# 2. Sprawdź odwrotnie - produkty "tynk/zaprawa/klej" w kategorii izolacje
izolacje_subs = ['welny','styropiany','plyty-xps','izolacje-budowlane']
plaster_words = ['tynk','zaprawa','klej','gips','grunt','gładź','szpachla','spoina']
wrong_cat2 = []
for slug in izolacje_subs:
    prods = prods_in_cat(slug, 300)
    for p in prods:
        if any(w.lower() in p['name'].lower() for w in plaster_words):
            wrong_cat2.append({'slug': slug, 'name': p['name'], 'id': p['_id']})

print(f"\nProdukty chemii budowlanej w izolacjach: {len(wrong_cat2)}")
for p in wrong_cat2[:20]:
    print(f"  [{p['slug']}] {p['name']}")

# 3. Sprawdź kategorię "kielnie" - ile ma produktów i czy jest na bechcicki.pl
kielnie = sq('count(*[_type=="product" && references(*[_type=="category" && slug.current=="kielnie"]._id) && !(name match "P-*")])')
print(f"\nProduktów w 'kielnie': {kielnie}")

# 4. Sprawdź puste kategorie L2
print("\nSprawdzam puste podkategorie L2...")
empty_cats = []
l2_cats = sq('*[_type=="category" && defined(parent) && !defined(parent->parent)]{"slug":slug.current, "name":name, "_id":_id}')
for cat in l2_cats:
    count = sq(f'count(*[_type=="product" && references("{cat["_id"]}") && !(name match "P-*")])')
    if isinstance(count, int) and count == 0:
        empty_cats.append(cat)
        
print(f"Puste podkategorie L2: {len(empty_cats)}")
for c in empty_cats:
    print(f"  {c['name']} ({c['slug']})")


Produkty izolacyjne w chemii budowlanej: 0



Produkty chemii budowlanej w izolacjach: 0

Produktów w 'kielnie': 1

Sprawdzam puste podkategorie L2...


Puste podkategorie L2: 40
  BHP (bhp)
  Dachy zielone (dachy-zielone)
  Dodatki do zapraw i betonu (dodatki-do-zapraw-i-betonu)
  Elementy mocujące uniwersalne (elementy-mocujace-uniwersalne)
  Farby specjalistyczne (farby-specjalistyczne)
  Farby do metalu (farby-do-metalu)
  Folie (folie)
  Fotowoltaika (fotowoltaika)
  Kleje (kleje)
  Elektronarzędzia (elektronarzedzia)
  Farby elewacyjne (farby-elewacyjne)
  Farby wewnętrzne (farby-wewnetrzne)
  Galanteria betonowa (galanteria-betonowa)
  Izolacje budowlane (izolacje-budowlane)
  Izolacje techniczne (izolacje-techniczne)
  Materiały konstrukcyjne (materialy-konstrukcyjne)
  Mocowania do suchej zabudowy (mocowania-do-suchej-zabudowy)
  Mocowania do sufitów podwieszanych (mocowania-do-sufitow-podwieszanych)
  Narożniki i listwy (narozniki-i-listwy)
  Narzędzia malarskie (narzedzia-malarskie)
  Panele ścienne i tapety (panele-scienne-i-tapety)
  Płytki ceramiczne (plytki-ceramiczne)
  Pokrycia dachowe (pokrycia-dachowe)
  Schody i akc

In [39]:

import re, json, requests

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
HEADERS = {"Authorization": f"Bearer {TOKEN}"}
BASE = "https://nzcwegq7.api.sanity.io/v2021-06-07/data/query/production"
MUTATE = "https://nzcwegq7.api.sanity.io/v2021-06-07/data/mutate/production"

def sq(q):
    r = requests.get(BASE, params={"query": q}, headers=HEADERS, timeout=30)
    return r.json().get("result", [])

def mutate(mutations):
    r = requests.post(MUTATE, headers=HEADERS, json={"mutations": mutations}, timeout=30)
    return r.status_code, r.json()

# ====== SPRAWDŹ PRODUKTY W L3 KATEGORIACH ======
# Może "wełna w tynkach" to produkty w L3 subcategoriach tynków?
print("=== Szukam wełny w podkategoriach tynków (L3) ===")
# Pobierz ID kategorii tynki
tynki_id = sq('*[_type=="category" && slug.current=="tynki"]._id[0]')
if tynki_id:
    # Znajdź wszystkie L3 subcategorie tynków
    tynki_subs = sq(f'*[_type=="category" && parent._ref=="{tynki_id}"]{{_id,name,"slug":slug.current}}')
    print(f"Podkategorie tynków: {[s['slug'] for s in tynki_subs]}")
    
    # Szukaj wełny w tych subcategoriach
    for sub in tynki_subs:
        prods = sq(f'*[_type=="product" && category._ref=="{sub["_id"]}" && !(name match "P-*")]{{_id,name}}[0..20]')
        izol_prods = [p for p in prods if any(w in p['name'].lower() for w in ['wełn','welna','styropian','rockwool','isover','ursa','mineralna'])]
        if izol_prods:
            print(f"\n  PROBLEM w [{sub['slug']}]:")
            for p in izol_prods:
                print(f"    {p['name']}")

# ====== SPRAWDŹ ODWROTNIE - TYNKI W IZOLACJACH ======
print("\n=== Szukam tynków/klejów w subcategoriach wełen ===")
welny_id = sq('*[_type=="category" && slug.current=="welny"]._id[0]')
if welny_id:
    welny_subs = sq(f'*[_type=="category" && parent._ref=="{welny_id}"]{{_id,name,"slug":slug.current}}')
    for sub in welny_subs:
        prods = sq(f'*[_type=="product" && category._ref=="{sub["_id"]}" && !(name match "P-*")]{{_id,name}}[0..50]')
        wrong = [p for p in prods if any(w in p['name'].lower() for w in ['tynk','klej','zaprawa','gips','farba','grunt'])]
        if wrong:
            print(f"\n  PROBLEM w [{sub['slug']}]:")
            for p in wrong:
                print(f"    {p['name']}")
        else:
            print(f"  OK: {sub['slug']} ({len(prods)} prods, brak tynków/klejów)")


=== Szukam wełny w podkategoriach tynków (L3) ===


Podkategorie tynków: []

=== Szukam tynków/klejów w subcategoriach wełen ===


In [43]:

import requests, json, re

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
BASE = "https://nzcwegq7.api.sanity.io/v2021-06-07/data/query/production"
MUTATE = "https://nzcwegq7.api.sanity.io/v2021-06-07/data/mutate/production"

def sq(q): return requests.get(BASE, params={"query": q}, headers=HEADERS, timeout=30).json().get("result", [])

def fix_spec_thickness(prod_id, correct_thickness_mm, extra_specs=None):
    """Napraw parametr Grubość w produkcie."""
    prod = sq(f'*[_id=="{prod_id}"]{{_id,name,technicalSpec}}[0]')
    if not prod: return False
    
    specs = prod.get('technicalSpec', []) or []
    updated = False
    new_specs = []
    for s in specs:
        label = s.get('label', '').lower()
        if any(k in label for k in ['grub', 'thick', 'grubość']):
            new_specs.append({'label': s['label'], 'value': str(correct_thickness_mm)})
            updated = True
        else:
            new_specs.append(s)
    
    if not updated:
        new_specs.insert(0, {'label': 'Grubość', 'value': str(correct_thickness_mm)})
    
    if extra_specs:
        for label, value in extra_specs.items():
            found = False
            for i, s in enumerate(new_specs):
                if s['label'].lower() == label.lower():
                    new_specs[i] = {'label': s['label'], 'value': str(value)}
                    found = True
                    break
            if not found:
                new_specs.append({'label': label, 'value': str(value)})
    
    mutations = [{"patch": {"id": prod_id, "set": {"technicalSpec": new_specs}}}]
    r = requests.post(MUTATE, headers=HEADERS, json={"mutations": mutations}, timeout=30)
    return r.status_code in [200, 201]

# ============================================================
# POTWIERDZONE NAPRAWY z bechcicki.pl
# ============================================================

with open('/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/dim_mismatches.json') as f:
    mismatches = json.load(f)

fixes = 0
print("=== Naprawiam produkty z potwierdzonymi danymi ===\n")

for m in mismatches:
    name = m['name'].lower()
    pid = m['_id']
    name_dims = m['name_dims']
    spec_dims = m['spec_dims']
    
    if not name_dims:
        continue
    
    # Wyciągnij grubość z nazwy - szukaj mm w kontekście wełny/styropianu/izolacji
    dim_val = name_dims[0]  # Pierwszy wymiar z nazwy
    
    # Sprawdź czy spec ma inną grubość dla produktów izolacyjnych
    is_insulation = any(k in name for k in ['welna','wełna','styropian','izolac','rockwool','isover','ursa','paroc'])
    is_tool = any(k in name for k in ['miara','szlifierka','wiertar','piła','poziomnic'])
    
    if is_insulation and name_dims and spec_dims and not set(name_dims) & set(spec_dims):
        # Grubość izolacji z nazwy jest wiążąca
        correct_mm = name_dims[0]
        # Sprawdź czy mamy mm czy cm w nazwie
        name_lower = m['name'].lower()
        if 'cm' in name_lower and f"{correct_mm}cm" in name_lower.replace(' ',''):
            correct_mm_val = int(float(correct_mm) * 10)
            unit_str = f"{correct_mm} cm ({correct_mm_val} mm)"
        else:
            correct_mm_val = correct_mm
            unit_str = f"{correct_mm} mm"
        
        ok = fix_spec_thickness(pid, correct_mm_val)
        status = "✅" if ok else "❌"
        print(f"{status} {m['name'][:60]}")
        print(f"   Było: {spec_dims} → Powinno być: {unit_str}")
        if ok: fixes += 1

print(f"\n=== Naprawiono: {fixes} produktów ===")


=== Naprawiam produkty z potwierdzonymi danymi ===



✅ Welna Mineralna Szklana Ursa Homewall 100 Mm
   Było: ['180'] → Powinno być: 100 mm


✅ Welna Mineralna Isover Uni Mata 039 2X80 5500 1200 Mm Mps
   Było: ['180'] → Powinno być: 1200 mm


✅ Alna Rockwool Ventirock Plus 034 050 01000 0600 Mm Opak 4 8 
   Było: ['50'] → Powinno być: 0600 mm


✅ Welna Mineralna Szklana Ursa Homewall 100 Mm
   Było: ['80'] → Powinno być: 100 mm


...

✅ Kolek Do Styropianu Klimas Wkret Met 300 Mm
   Było: ['260'] → Powinno być: 300 mm


✅ Welna Mineralna Isover Uni Mata 039 2X80 5500 1200 Mm Mps
   Było: ['200'] → Powinno być: 1200 mm


✅ Plyta Izolacyjna Termo Organika Pir Etx 30 600 1200 Mm
   Było: ['80'] → Powinno być: 1200 mm


✅ Kolek Do Styropianu Klimas Wkret Met 300 Mm
   Było: ['200'] → Powinno być: 300 mm


✅ Welna Mineralna Ursa Pureone 31 100 1200 4000 Mm Opak 4 8 M
   Było: ['100'] → Powinno być: 4000 mm

=== Naprawiono: 43 produktów ===


In [47]:

import requests, json, re

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
BASE = "https://nzcwegq7.api.sanity.io/v2021-06-07/data/query/production"
MUTATE = "https://nzcwegq7.api.sanity.io/v2021-06-07/data/mutate/production"

def sq(q): return requests.get(BASE, params={"query": q}, headers=HEADERS, timeout=30).json().get("result", [])
def mutate(m): 
    r = requests.post(MUTATE, headers=HEADERS, json={"mutations": m}, timeout=30)
    return r.status_code

with open('/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/dim_mismatches.json') as f:
    mismatches = json.load(f)

# Załaduj wszystkie ID z mismatches które były izolacyjne (właśnie zmienione)
changed_ids = []
for m in mismatches:
    name = m['name'].lower()
    if any(k in name for k in ['welna','wełna','styropian','rockwool','isover','ursa','paroc','kolek']):
        changed_ids.append(m['_id'])

print(f"Produktów do cofnięcia: {len(changed_ids)}")

# Dla każdego - pobierz obecny stan i cofnij grubość do wartości z orig spec_dims
reverted = 0
failed = 0
for i, m in enumerate([x for x in mismatches if x['_id'] in changed_ids]):
    pid = m['_id']
    spec_dims = m['spec_dims']  # ORYGINALNE wymiary z spec (przed moją zmianą)
    
    if not spec_dims:
        continue
    
    # Pobierz obecny stan
    prod = sq(f'*[_id=="{pid}"]{{_id,name,technicalSpec}}[0]')
    if not prod:
        continue
    
    specs = prod.get('technicalSpec', []) or []
    new_specs = []
    reverted_this = False
    
    for s in specs:
        label = s.get('label', '').lower()
        if any(k in label for k in ['grub', 'thick', 'grubość']) and not reverted_this:
            # Przywróć ORYGINALNĄ wartość (pierwsze spec_dim)
            new_specs.append({'label': s['label'], 'value': str(spec_dims[0])})
            reverted_this = True
        else:
            new_specs.append(s)
    
    if reverted_this:
        status = mutate([{"patch": {"id": pid, "set": {"technicalSpec": new_specs}}}])
        if status in [200, 201]:
            reverted += 1
        else:
            failed += 1

print(f"\nCofnięto: {reverted} | Błędy: {failed}")
print("\n=== REVERT ZAKOŃCZONY ===")
print("Parametry przywrócone do stanu PRZED błędną zmianą.")


Produktów do cofnięcia: 42



Cofnięto: 42 | Błędy: 0

=== REVERT ZAKOŃCZONY ===
Parametry przywrócone do stanu PRZED błędną zmianą.


In [51]:

import requests

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
BASE  = "https://nzcwegq7.api.sanity.io/v2021-06-07/data/query/production"
MUTATE = "https://nzcwegq7.api.sanity.io/v2021-06-07/data/mutate/production"

def sq(q): return requests.get(BASE, params={"query": q}, headers=HEADERS, timeout=30).json().get("result", [])

# 1. Pobierz ID kategorii oswietlenie i jej podkategorii
cats = sq('*[_type=="category" && (slug.current=="oswietlenie" || parent->slug.current=="oswietlenie" || parent->parent->slug.current=="oswietlenie")]{_id, name, "slug":slug.current, "parentSlug":parent->slug.current}')
print(f"Kategorie do usunięcia ({len(cats)}):")
for c in cats:
    print(f"  {c['name']} ({c['slug']}) parent={c.get('parentSlug','—')}")

# 2. Sprawdź czy żaden produkt nie wskazuje na te kategorie
ids = [c['_id'] for c in cats]
for cat_id in ids:
    count = sq(f'count(*[_type=="product" && category._ref=="{cat_id}"])')
    print(f"  Produkty w {cat_id}: {count}")


Kategorie do usunięcia (1):
  Oświetlenie (oswietlenie) parent=None
  Produkty w cat-oswietlenie: 0


In [55]:

import requests
TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
MUTATE = "https://nzcwegq7.api.sanity.io/v2021-06-07/data/mutate/production"

# Usuń kategorię oswietlenie (ID: cat-oswietlenie)
mutations = [{"delete": {"id": "cat-oswietlenie"}}]
r = requests.post(MUTATE, headers=HEADERS, json={"mutations": mutations}, timeout=30)
print(f"Status: {r.status_code}")
print(r.json())


Status: 200
{'transactionId': 'bM4Vn6Xhr0kQ8myfjD1fTy', 'results': [{'operation': 'delete'}]}


In [59]:

import requests, json
TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
BASE  = "https://nzcwegq7.api.sanity.io/v2021-06-07/data/query/production"
MUTATE = "https://nzcwegq7.api.sanity.io/v2021-06-07/data/mutate/production"

def sq(q): return requests.get(BASE, params={"query": q}, headers=HEADERS, timeout=30).json().get("result")

prod = sq('*[_type=="product" && slug.current=="adolin-superdeck-do-tarasow-i-mebli-ogrodowych-antracyt-2-5-l"][0]{_id,name,ean,sku,"imgRef":mainImage.asset->_id,"imgUrl":mainImage.asset->url,technicalSpec,shortDescription}')
print(json.dumps(prod, indent=2, ensure_ascii=False))


{
  "_id": "product-p0001804",
  "ean": "5904078204962",
  "imgRef": null,
  "imgUrl": null,
  "name": "Adolin Superdeck Do Tarasow I Mebli Ogrodowych Antracyt 2 5 L",
  "shortDescription": "Wodoodporny olej do ochrony i dekoracji drewna tarasowego. Zapewnia długotrwałą ochronę przed warunkami atmosferycznymi i promieniowaniem UV. Dostępny w odcieniu szarym.",
  "sku": "P-0001804",
  "technicalSpec": [
    {
      "label": "Typ",
      "value": "Olej do drewna"
    },
    {
      "label": "Zastosowanie",
      "value": "Drewno tarasowe, elewacje drewniane"
    },
    {
      "label": "Opakowanie",
      "value": "2,5 L"
    },
    {
      "label": "Marka",
      "value": "Sadolin"
    },
    {
      "label": "Kolor",
      "value": "Szary"
    }
  ]
}


In [63]:

import requests, json

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
MUTATE = "https://nzcwegq7.api.sanity.io/v2021-06-07/data/mutate/production"

# 1. Upload zdjęcia
img_path = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/tmp_images/sadolin-superdeck-antracyt_1.jpeg"
with open(img_path, "rb") as f:
    img_data = f.read()

upload_resp = requests.post(
    "https://nzcwegq7.api.sanity.io/v2021-06-07/assets/images/production",
    data=img_data,
    headers={"Authorization": f"Bearer {TOKEN}", "Content-Type": "image/jpeg"},
    timeout=60
)
print(f"Upload status: {upload_resp.status_code}")
asset_id = upload_resp.json().get("document", {}).get("_id")
print(f"Asset ID: {asset_id}")

# 2. Popraw produkt - nazwa, opis, specyfikacje, zdjęcie
patch = {
    "name": "Sadolin Superdeck Olej Do Tarasów I Mebli Ogrodowych Antracyt 2,5 L",
    "mainImage": {"_type": "image", "asset": {"_type": "reference", "_ref": asset_id}},
    "shortDescription": "Nowoczesny olej z żywicami do ochrony i dekoracji tarasów drewnianych oraz mebli ogrodowych. Hydrofobowa formuła odpycha wodę, głęboko wnika w drewno i trwale barwi na kolor antracytowy. Nie łuszczy się.",
    "description": "Sadolin Superdeck to nowoczesny olej zawierający naturalne żywice, stworzony do ochrony i dekoracji drewnianych tarasów, mebli ogrodowych oraz innych powierzchni drewnianych na zewnątrz. Produkt głęboko wnika w strukturę drewna, odżywiając je i wzmacniając. Dzięki hydrofobowej formule skutecznie odpycha wodę, zapewniając długotrwałą ochronę przed czynnikami atmosferycznymi i promieniowaniem UV. Nie tworzy powłoki – pozostawia drewno naturalne w dotyku. Kolor antracytowy trwale barwi drewno, podkreślając jego naturalny rysunek. Wydajność: do 18 m²/l na drewnie twardym.",
    "technicalSpec": [
        {"label": "Marka",        "value": "Sadolin"},
        {"label": "Typ",          "value": "Olej ochronno-dekoracyjny do drewna"},
        {"label": "Pojemność",    "value": "2,5 L"},
        {"label": "Kolor",        "value": "Antracyt"},
        {"label": "Zastosowanie", "value": "Tarasy drewniane, meble ogrodowe, drewno zewnętrzne"},
        {"label": "Baza",         "value": "Olej z żywicami"},
        {"label": "Wykończenie",  "value": "Mat"},
        {"label": "Wydajność",    "value": "do 18 m²/l (drewno twarde), do 14 m²/l (drewno miękkie)"},
        {"label": "Warstwy",      "value": "2–3 warstwy co 24h"},
        {"label": "Opakowanie",   "value": "Puszka metalowa"},
    ]
}

mutations = [{"patch": {"id": "product-p0001804", "set": patch}}]
r = requests.post(MUTATE, headers=HEADERS, json={"mutations": mutations}, timeout=30)
print(f"Mutate status: {r.status_code}")
print(r.json().get("transactionId", r.text[:200]))


Upload status: 200
Asset ID: image-410b1d5838a26af2ead65f13014db4623682d8d1-1200x1200-jpg


Mutate status: 200
UAjCtlAuHSg7wG7TzI6UKu


In [67]:

import requests
TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
HEADERS = {"Authorization": f"Bearer {TOKEN}"}
BASE = "https://nzcwegq7.api.sanity.io/v2021-06-07/data/query/production"
def sq(q): return requests.get(BASE, params={"query": q}, headers=HEADERS, timeout=30).json().get("result")

prod = sq('*[_id=="product-p0001804"][0]{name,"imgUrl":mainImage.asset->url,shortDescription,technicalSpec[0..3]}')
print(f"Nazwa:  {prod['name']}")
print(f"Zdjęcie: {prod['imgUrl'][:80] if prod.get('imgUrl') else '❌ BRAK'}")
print(f"Opis:   {prod['shortDescription'][:80]}")
print(f"Spec:   {[(s['label'],s['value']) for s in prod['technicalSpec']]}")


Nazwa:  Sadolin Superdeck Olej Do Tarasów I Mebli Ogrodowych Antracyt 2,5 L
Zdjęcie: https://cdn.sanity.io/images/nzcwegq7/production/410b1d5838a26af2ead65f13014db46
Opis:   Nowoczesny olej z żywicami do ochrony i dekoracji tarasów drewnianych oraz mebli
Spec:   [('Marka', 'Sadolin'), ('Typ', 'Olej ochronno-dekoracyjny do drewna'), ('Pojemność', '2,5 L'), ('Kolor', 'Antracyt')]


In [71]:

import requests, re, json
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0 Safari/537.36"}

def get_product_urls_from_category(url):
    """Pobierz listę URL produktów ze strony kategorii bechcicki.pl"""
    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        soup = BeautifulSoup(r.text, 'html.parser')
        # Szukaj linków do produktów - wzorzec /produkt-nazwa-id-p-XXXXXXX
        links = soup.find_all('a', href=re.compile(r'/[a-z0-9-]+-id-p-\d+'))
        urls = list(dict.fromkeys([a['href'] if a['href'].startswith('http') else 'https://www.bechcicki.pl' + a['href'] for a in links]))
        # Sprawdź paginację
        pages = soup.find_all('a', href=re.compile(r'[?&]page=\d+'))
        max_page = 1
        for p in pages:
            m = re.search(r'page=(\d+)', p['href'])
            if m: max_page = max(max_page, int(m.group(1)))
        return urls, max_page
    except Exception as e:
        return [], 1

# TEST: kategoria wełny fasadowe
test_url = "https://www.bechcicki.pl/izolacje/welny/welny-fasadowe/"
urls, pages = get_product_urls_from_category(test_url)
print(f"Kategoria: {test_url}")
print(f"Produktów na stronie 1: {len(urls)}, stron: {pages}")
for u in urls[:5]:
    print(f"  {u}")


Kategoria: https://www.bechcicki.pl/izolacje/welny/welny-fasadowe/
Produktów na stronie 1: 0, stron: 1
